# Mask R-CNN - Train oilwell Model

In [1]:
import os
import sys
import datetime
import random
import skimage.draw
from skimage.measure import label
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
# Root directory of the project
ROOT_DIR = os.path.abspath("../../")
sys.path.append(ROOT_DIR)


import mrcnn.model as modellib
from mrcnn import utils
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import TensorBoard
from mrcnn import visualize, utils

%matplotlib inline 

# Directory to save logs and trained model
MODEL_DIR = os.path.join(ROOT_DIR, "posIOU/logs")



In [2]:
from mrcnn.config import Config

In [3]:
DATASET_DIR = os.path.join(ROOT_DIR, "datasets/siteA_oilwell")
#LOGS_DIR = os.path.join(ROOT_DIR, "logs")
COCO_WEIGHTS_PATH = os.path.join(ROOT_DIR, "mask_rcnn_coco.h5")

In [4]:
# class OilwellConfig(Config):
#     """Configuration for training on the oilwell  dataset.
#     Derives from the base Config class and overrides some values.
#     """
#     # Give the configuration a recognizable name
#     NAME = "oilwell"

    
#     IMAGES_PER_GPU = 2

#     # Number of classes (including background)
#     NUM_CLASSES = 1 + 1  # Background + balloon

#     # Number of training steps per epoch
#     STEPS_PER_EPOCH = 100

#     # Skip detections with < 90% confidence
#     DETECTION_MIN_CONFIDENCE = 0.8

In [5]:
class OilwellConfig(Config):
    NAME = "oilwell"
    NUM_CLASSES = 1 + 1  # background + oilwell
    GPU_COUNT = 1
    IMAGES_PER_GPU = 1
    IMAGE_MIN_DIM = 512
    IMAGE_MAX_DIM = 512
    IMAGE_RESIZE_MODE = "none"
    STEPS_PER_EPOCH = 100
    DETECTION_MIN_CONFIDENCE = 0.9

In [6]:
class OilWellDataset(utils.Dataset):
    def load_oilwells(self, dataset_dir, subset):
        """Load a subset of the OilWell dataset."""
        # Add class
        self.add_class("oilwell", 1, "oilwell")
        
        # Path to images and masks
        assert subset in ["train", "val", "test"]
        subset_dir = os.path.join(dataset_dir, subset)
        image_dir = os.path.join(subset_dir, "images")
        mask_dir = os.path.join(subset_dir, "masks")
        
        # List image files
        for filename in os.listdir(image_dir):
            if not filename.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff")):
                continue
            image_path = os.path.join(image_dir, filename)
            mask_path = os.path.join(mask_dir, filename)  # assumes mask has same name

            # Read image to get size
            image = skimage.io.imread(image_path)
            height, width = image.shape[:2]

            self.add_image(
                "oilwell",
                image_id=filename,
                path=image_path,
                width=width, height=height,
                mask_path=mask_path
            )

    def load_mask(self, image_id):
        """Generate instance masks for an image."""
        image_info = self.image_info[image_id]
        if image_info["source"] != "oilwell":
            return super(self.__class__, self).load_mask(image_id)
        
        # Read the saved raster mask (0 background, >0 well pixels)
        info = self.image_info[image_id]
        m = skimage.io.imread(info["mask_path"])
        if m.ndim == 3:
            m = m[..., 0]

        # Convert to binary
        m = (m > 0)

        # Split into instances using connected components
        labeled = label(m, connectivity=2)  # 8-connected
        instance_ids = np.unique(labeled)
        instance_ids = instance_ids[instance_ids != 0]  # remove background

        # If no wells in this tile
        if instance_ids.size == 0:
            mask = np.zeros([info["height"], info["width"], 0], dtype=bool)
            class_ids = np.array([], dtype=np.int32)
            return mask, class_ids

        # Build instance mask stack: [H, W, instance_count]
        mask = np.zeros([info["height"], info["width"], instance_ids.size], dtype=np.uint8)
        for i, inst_id in enumerate(instance_ids):
            mask[:, :, i] = (labeled == inst_id).astype(np.uint8)

        # Return boolean masks + class IDs (all 1 because one class)
        return mask.astype(bool), np.ones([mask.shape[-1]], dtype=np.int32)


def image_reference(self, image_id):
    """Return the path of the image."""
    info = self.image_info[image_id]

    if info["source"] == "oilwell":
        return info["path"]
    else:
        return super(self.__class__, self).image_reference(image_id)

In [7]:
config = OilwellConfig()
config.display()


Configurations:
BACKBONE                       resnet101
BACKBONE_STRIDES               [4, 8, 16, 32, 64]
BATCH_SIZE                     1
BBOX_STD_DEV                   [0.1 0.1 0.2 0.2]
COMPUTE_BACKBONE_SHAPE         None
DETECTION_MAX_INSTANCES        35
DETECTION_MIN_CONFIDENCE       0.9
DETECTION_NMS_THRESHOLD        0.3
FPN_CLASSIF_FC_LAYERS_SIZE     1024
GPU_COUNT                      1
GRADIENT_CLIP_NORM             5.0
IMAGES_PER_GPU                 1
IMAGE_CHANNEL_COUNT            3
IMAGE_MAX_DIM                  512
IMAGE_META_SIZE                14
IMAGE_MIN_DIM                  512
IMAGE_MIN_SCALE                0
IMAGE_RESIZE_MODE              none
IMAGE_SHAPE                    [512 512   3]
LEARNING_MOMENTUM              0.9
LEARNING_RATE                  0.001
LOSS_WEIGHTS                   {'rpn_class_loss': 1.0, 'rpn_bbox_loss': 1.0, 'mrcnn_class_loss': 1.0, 'mrcnn_bbox_loss': 1.0, 'mrcnn_mask_loss': 1.0}
MASK_POOL_SIZE                 14
MASK_SHAPE                

In [8]:
model = modellib.MaskRCNN(mode="training", config=config, model_dir=MODEL_DIR)

In [9]:
model.load_weights(COCO_WEIGHTS_PATH, by_name=True, exclude=[
            "mrcnn_class_logits", "mrcnn_bbox_fc",
            "mrcnn_bbox", "mrcnn_mask"])

In [10]:
# Training dataset.
dataset_train = OilWellDataset()
dataset_train.load_oilwells(DATASET_DIR, "train")
dataset_train.prepare()

# Validation dataset
dataset_val = OilWellDataset()
dataset_val.load_oilwells(DATASET_DIR, "val")
dataset_val.prepare()

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,              # stop if no improvement for 5 epochs
    restore_best_weights=True
)

log_dir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))

tensorboard = TensorBoard(
    log_dir=log_dir,
    histogram_freq=0,
    write_graph=True,
    update_freq='epoch'
)

In [11]:
#Training network heads
model.train(dataset_train, dataset_val,
            learning_rate=config.LEARNING_RATE,
            epochs=30,
            layers='heads',
            custom_callbacks=[tensorboard])


Starting at epoch 0. LR=0.001

Checkpoint Path: c:\oil_well_n\Code\Mask-RCNN_TF2.14.0\posIOU/logs\oilwell20260521T0643\mask_rcnn_oilwell_{epoch:04d}.h5
Selecting layers to train
fpn_c5p5               (Conv2D)
fpn_c4p4               (Conv2D)
fpn_c3p3               (Conv2D)
fpn_c2p2               (Conv2D)
fpn_p5                 (Conv2D)
fpn_p2                 (Conv2D)
fpn_p3                 (Conv2D)
fpn_p4                 (Conv2D)
rpn_model              (Functional)
mrcnn_mask_conv1       (TimeDistributed)
mrcnn_mask_bn1         (TimeDistributed)
mrcnn_mask_conv2       (TimeDistributed)
mrcnn_mask_bn2         (TimeDistributed)
mrcnn_class_conv1      (TimeDistributed)
mrcnn_class_bn1        (TimeDistributed)
mrcnn_mask_conv3       (TimeDistributed)
mrcnn_mask_bn3         (TimeDistributed)
mrcnn_class_conv2      (TimeDistributed)
mrcnn_class_bn2        (TimeDistributed)
mrcnn_mask_conv4       (TimeDistributed)
mrcnn_mask_bn4         (TimeDistributed)
mrcnn_bbox_fc          (TimeDistribut

c:\Users\iaadeniran\AppData\Local\ESRI\conda\envs\maskrcnn_env\lib\site-packages\keras\src\optimizers\legacy\gradient_descent.py:114: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super().__init__(name, **kwargs)


Epoch 1/30
100/100 [==============================] - ETA: 0s - batch: 49.5000 - size: 1.0000 - loss: 3.4569 - rpn_class_loss: 0.5973 - rpn_bbox_loss: 1.5630 - mrcnn_class_loss: 0.1287 - mrcnn_bbox_loss: 0.5921 - mrcnn_mask_loss: 0.5759

c:\Users\iaadeniran\AppData\Local\ESRI\conda\envs\maskrcnn_env\lib\site-packages\keras\src\engine\training_v1.py:2335: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates = self.state_updates


100/100 [==============================] - 393s 4s/step - batch: 49.5000 - size: 1.0000 - loss: 3.4569 - rpn_class_loss: 0.5973 - rpn_bbox_loss: 1.5630 - mrcnn_class_loss: 0.1287 - mrcnn_bbox_loss: 0.5921 - mrcnn_mask_loss: 0.5759 - val_loss: 2.7305 - val_rpn_class_loss: 0.2592 - val_rpn_bbox_loss: 1.3003 - val_mrcnn_class_loss: 0.1454 - val_mrcnn_bbox_loss: 0.5119 - val_mrcnn_mask_loss: 0.5138
Epoch 2/30
100/100 [==============================] - 385s 4s/step - batch: 49.5000 - size: 1.0000 - loss: 2.3575 - rpn_class_loss: 0.1538 - rpn_bbox_loss: 1.2357 - mrcnn_class_loss: 0.0804 - mrcnn_bbox_loss: 0.4035 - mrcnn_mask_loss: 0.4841 - val_loss: 2.1049 - val_rpn_class_loss: 0.1128 - val_rpn_bbox_loss: 1.0798 - val_mrcnn_class_loss: 0.0727 - val_mrcnn_bbox_loss: 0.3558 - val_mrcnn_mask_loss: 0.4836
Epoch 3/30
100/100 [==============================] - 378s 4s/step - batch: 49.5000 - size: 1.0000 - loss: 2.0305 - rpn_class_loss: 0.1114 - rpn_bbox_loss: 1.0840 - mrcnn_class_loss: 0.0592 - m

In [12]:
class InferenceConfig(OilwellConfig):  
    GPU_COUNT = 1
    IMAGES_PER_GPU = 1  # inference on one image at a time

In [13]:
inference_config = InferenceConfig()

model_infer = modellib.MaskRCNN(
    mode="inference",
    config=inference_config,
    model_dir=log_dir
)

# Load the best weights from training
best_model_path = model.find_last()  # or your saved weights file
# best_model_path = r"C:\oil_well_n\Code\Mask-RCNN_TF2.14.0\new30\logs\oilwell20260518T1609\mask_rcnn_oilwell_0030.h5"
model_infer.load_weights(best_model_path, by_name=True)

Instructions for updating:
Use fn_output_signature instead
Re-starting from epoch 30


In [19]:
def _ensure_rgb3(image):
    """Ensure image is HxWx3 for Mask R-CNN."""
    if image.ndim == 2:
        image = np.stack([image]*3, axis=-1)
    elif image.ndim == 3 and image.shape[2] == 1:
        image = np.repeat(image, 3, axis=2)
    return image

def _union_mask(mask_3d, H, W):
    """[H,W,N] -> [H,W] boolean union mask"""
    if mask_3d is None or mask_3d.size == 0:
        return np.zeros((H, W), dtype=bool)
    if mask_3d.ndim == 2:
        return mask_3d.astype(bool)
    return (np.sum(mask_3d, axis=-1) > 0)

def save_overlays_and_metrics(model, dataset, out_dir, score_thresh=0.8, limit=None):
    os.makedirs(out_dir, exist_ok=True)

    # separate folders by TP/FP/FN/TN
    for sub in ["TP", "FP", "FN", "TN"]:
        os.makedirs(os.path.join(out_dir, sub), exist_ok=True)

    ids = dataset.image_ids[:limit] if limit else dataset.image_ids

    TP = FP = FN = TN = 0

    for image_id in tqdm(ids):
        image = dataset.load_image(image_id)
        image = _ensure_rgb3(image)
        H, W = image.shape[:2]

        # GET REAL FILENAME
        image_info = dataset.image_info[image_id]
        filename = os.path.splitext(os.path.basename(image_info["path"]))[0]

        gt_mask, _ = dataset.load_mask(image_id)
        gt_any = bool(gt_mask.size > 0 and np.sum(gt_mask) > 0)
        gt_u = _union_mask(gt_mask, H, W)

        r = model.detect([image], verbose=0)[0]

        # filter by threshold
        if r["scores"].size:
            keep = r["scores"] >= score_thresh
            pred_masks = r["masks"][:, :, keep] if np.any(keep) else np.zeros((H, W, 0), dtype=bool)
        else:
            pred_masks = np.zeros((H, W, 0), dtype=bool)

        pred_any = pred_masks.shape[-1] > 0
        pr_u = _union_mask(pred_masks, H, W)

        # confusion
        if gt_any and pred_any:
            TP += 1; status = "TP"
        elif (not gt_any) and pred_any:
            FP += 1; status = "FP"
        elif gt_any and (not pred_any):
            FN += 1; status = "FN"
        else:
            TN += 1; status = "TN"

        # save overlay image
        fig = plt.figure(figsize=(12, 4))

        ax1 = fig.add_subplot(1, 3, 1)
        ax1.imshow(image); ax1.set_title("Image"); ax1.axis("off")

        ax2 = fig.add_subplot(1, 3, 2)
        ax2.imshow(image)
        ax2.imshow(np.ma.masked_where(~gt_u, gt_u), alpha=0.5)
        ax2.set_title("GT"); ax2.axis("off")

        ax3 = fig.add_subplot(1, 3, 3)
        ax3.imshow(image)
        ax3.set_title(f"Pred ({status})") 
        ax3.axis("off")

        rois = r["rois"]
        masks = r["masks"]
        class_ids = r["class_ids"]
        scores = r["scores"]    

        for i in range(rois.shape[0]):
            if scores[i] < score_thresh:
                continue

            y1, x1, y2, x2 = rois[i]

            # mask overlay
            mask = masks[:, :, i]
            ax3.imshow(np.ma.masked_where(~mask, mask), alpha=0.4)

            # label text (like visualize.display_instances)
            label = dataset.class_names[class_ids[i]]
            score = scores[i]

            ax3.text(
                x1, y1,
                f"{label} {score:.2f}",
                color="white",
                fontsize=8,
                bbox=dict(facecolor="red", alpha=0.5, pad=2)
            )

        plt.tight_layout()
        out_path = os.path.join(out_dir, status, f"{filename}.png")
        
        plt.savefig(out_path, dpi=120)
        plt.close(fig)

    # metrics
    total = TP + FP + FN + TN
    accuracy  = (TP + TN) / total if total else 0.0
    precision = TP / (TP + FP) if (TP + FP) else 0.0
    recall    = TP / (TP + FN) if (TP + FN) else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    print("\nConfusion (image-level presence/absence):")
    print(f"TP={TP}  FP={FP}  FN={FN}  TN={TN}  (Total={total})")

    print("\nMetrics:")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1       : {f1:.4f}")
    print(f"Saved overlays to: {out_dir}")

    return {"TP": TP, "FP": FP, "FN": FN, "TN": TN,
            "accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

In [18]:
def compute_metrics(model_infer, dataset_test):
    image_ids = dataset_test.image_ids

    APs = []
    AP50s = []
    AP75s = []
    IoUs = []

    empty_images = 0
    false_positives = 0

    for image_id in image_ids:

        # Load image and GT
        image, image_meta, gt_class_id, gt_bbox, gt_mask = \
            modellib.load_image_gt(dataset_test, config, image_id)
        
        # Run inference
        results = model_infer.detect([image], verbose=0)
        r = results[0]

        num_gt = len(gt_class_id)
        num_pred = len(r["class_ids"])

        # Track empty images
        if num_gt == 0:
            empty_images += 1

        # CASE 1: No GT and No Prediction -> True Negative

        if num_gt == 0 and num_pred == 0:

            AP50s.append(1.0)
            AP75s.append(1.0)
            APs.append(1.0)
            IoUs.append(1.0)

        # CASE 2: No GT but Prediction exists -> False Positive

        elif num_gt == 0 and num_pred > 0:

            false_positives += 1

            AP50s.append(0.0)
            AP75s.append(0.0)
            APs.append(0.0)
            IoUs.append(0.0)

        # CASE 3: GT exists but no Prediction -> False Negative

        elif num_gt > 0 and num_pred == 0:

            AP50s.append(0.0)
            AP75s.append(0.0)
            APs.append(0.0)
            IoUs.append(0.0)
        
        # CASE 4: Normal detection evaluation

        else:

            # AP @ 0.50
            AP50, precisions, recalls, overlaps = utils.compute_ap(
                gt_bbox,
                gt_class_id,
                gt_mask,
                r["rois"],
                r["class_ids"],
                r["scores"],
                r["masks"],
                iou_threshold=0.5
            )

            # AP @ 0.75
            AP75, _, _, _ = utils.compute_ap(
                gt_bbox,
                gt_class_id,
                gt_mask,
                r["rois"],
                r["class_ids"],
                r["scores"],
                r["masks"],
                iou_threshold=0.75
            )

            # AP @ 1.00 (perfect overlap only)
            AP, _, _, _ = utils.compute_ap(
                gt_bbox,
                gt_class_id,
                gt_mask,
                r["rois"],
                r["class_ids"],
                r["scores"],
                r["masks"],
                iou_threshold=1.0
            )

            AP50s.append(AP50)
            AP75s.append(AP75)
            APs.append(AP)

            # Mean IoU
            if overlaps.size > 0:
                best_ious = overlaps.max(axis=1)
                IoUs.append(np.mean(best_ious))
            else:
                IoUs.append(0.0)

    # Final Metrics

    mAP50 = np.mean(AP50s)
    mAP75 = np.mean(AP75s)
    mAP = np.mean(APs)
    mean_IoU = np.mean(IoUs)

    # False Positive Rate on empty images
    fp_rate = false_positives / empty_images if empty_images > 0 else 0

    print(f"mAP@50   : {mAP50:.3f}")
    print(f"mAP@75   : {mAP75:.3f}")
    print(f"mAP  : {mAP:.3f}")
    print(f"Mean IoU : {mean_IoU:.3f}")
    print(f"FP Rate  : {fp_rate:.3f}")




In [20]:
out_dir=r"C:\overlays_pr"

In [21]:
data_dir = r"C:\oil_well_n\Code\Mask-RCNN_TF2.14.0\datasets\siteA_oilwell"
dataset_val = OilWellDataset()
dataset_val.load_oilwells(data_dir, "val")
dataset_val.prepare()

In [22]:
object_level_metrics_val = compute_metrics(model_infer, dataset_val)

mAP@50   : 0.422
mAP@75   : 0.202
mAP  : 0.200
Mean IoU : 0.725
FP Rate  : 0.000


In [26]:
data_dir = r"C:\oil_well_n\Code\Mask-RCNN_TF2.14.0\datasets\siteA_oilwell"
dataset_test = OilWellDataset()
dataset_test.load_oilwells(data_dir, "test")
dataset_test.prepare()

In [27]:
image_level_metrics_test = save_overlays_and_metrics(
    model=model_infer,
    dataset=dataset_test,
    out_dir=os.path.join(out_dir, "30siteA_test"),
    score_thresh=0.8
)

100%|██████████| 202/202 [02:31<00:00,  1.34it/s]


Confusion (image-level presence/absence):
TP=154  FP=1  FN=0  TN=47  (Total=202)

Metrics:
Accuracy : 0.9950
Precision: 0.9935
Recall   : 1.0000
F1       : 0.9968
Saved overlays to: C:\overlays_pr\30siteA_test


In [28]:
object_level_metrics_test = compute_metrics(model_infer, dataset_test)

mAP@50   : 0.446
mAP@75   : 0.235
mAP  : 0.233
Mean IoU : 0.732
FP Rate  : 0.021


In [29]:
data_dir = r"C:\oil_well_n\Code\Mask-RCNN_TF2.14.0\datasets\SiteA_pre"
dataset_test = OilWellDataset()
dataset_test.load_oilwells(data_dir, "test")
dataset_test.prepare()

In [31]:
object_level_metrics_test = compute_metrics(model_infer, dataset_test)

mAP@50   : 0.455
mAP@75   : 0.248
mAP  : 0.248
Mean IoU : 0.736
FP Rate  : 0.000


In [32]:
metrics_test = save_overlays_and_metrics(
    model=model_infer,
    dataset=dataset_test,
    out_dir=os.path.join(out_dir, "30siteApre_test"),
    score_thresh=0.8
)

100%|██████████| 101/101 [01:12<00:00,  1.40it/s]


Confusion (image-level presence/absence):
TP=76  FP=0  FN=0  TN=25  (Total=101)

Metrics:
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1       : 1.0000
Saved overlays to: C:\overlays_pr\30siteApre_test


In [33]:
data_dir = r"C:\oil_well_n\Code\Mask-RCNN_TF2.14.0\datasets\SiteA_post"
dataset_test = OilWellDataset()
dataset_test.load_oilwells(data_dir, "test")
dataset_test.prepare()

In [34]:
object_level_metrics_test = compute_metrics(model_infer, dataset_test)

mAP@50   : 0.437
mAP@75   : 0.221
mAP  : 0.218
Mean IoU : 0.727
FP Rate  : 0.043


In [35]:
metrics_test = save_overlays_and_metrics(
    model=model_infer,
    dataset=dataset_test,
    out_dir=os.path.join(out_dir, "30siteApost_test"),
    score_thresh=0.8
)

100%|██████████| 101/101 [01:13<00:00,  1.38it/s]


Confusion (image-level presence/absence):
TP=78  FP=1  FN=0  TN=22  (Total=101)

Metrics:
Accuracy : 0.9901
Precision: 0.9873
Recall   : 1.0000
F1       : 0.9936
Saved overlays to: C:\overlays_pr\30siteApost_test


In [36]:
data_dir = r"C:\oil_well_n\Code\Mask-RCNN_TF2.14.0\datasets\siteB_oilwell"
dataset_test = OilWellDataset()
dataset_test.load_oilwells(data_dir, "test")
dataset_test.prepare()

In [37]:
object_level_metrics_test = compute_metrics(model_infer, dataset_test)

mAP@50   : 0.235
mAP@75   : 0.218
mAP  : 0.218
Mean IoU : 0.250
FP Rate  : 0.000


In [39]:
metrics_test = save_overlays_and_metrics(
    model=model_infer,
    dataset=dataset_test,
    out_dir=os.path.join(out_dir, "30siteBpost_test"),
    score_thresh=0.8
)

100%|██████████| 101/101 [01:09<00:00,  1.45it/s]


Confusion (image-level presence/absence):
TP=5  FP=0  FN=74  TN=22  (Total=101)

Metrics:
Accuracy : 0.2673
Precision: 1.0000
Recall   : 0.0633
F1       : 0.1190
Saved overlays to: C:\overlays_pr\30siteBpost_test


In [16]:
# all validation data
image_ids = dataset_val.image_ids

APs = []

for image_id in image_ids:
    # Load image and ground truth
    image, image_meta, gt_class_id, gt_bbox, gt_mask = \
        modellib.load_image_gt(dataset_val, config, image_id)
   
    # Skip images with no ground truth
    if len(gt_class_id) == 0:
        continue

    # Run detection
    results = model_infer.detect([image], verbose=0)
    r = results[0]

    # Compute AP
    AP, precisions, recalls, overlaps = \
        utils.compute_ap(gt_bbox, gt_class_id, gt_mask,
                         r["rois"], r["class_ids"], r["scores"], r["masks"])

    APs.append(AP)

# Mean Average Precision
mAP = np.mean(APs)
print("mAP: ", mAP)

c:\Users\iaadeniran\AppData\Local\ESRI\conda\envs\maskrcnn_env\lib\site-packages\keras\src\engine\training_v1.py:2359: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


mAP:  0.28090015491470693


In [14]:
def _ensure_rgb3(image):
    """Ensure image is HxWx3 for Mask R-CNN."""
    if image.ndim == 2:
        image = np.stack([image]*3, axis=-1)
    elif image.ndim == 3 and image.shape[2] == 1:
        image = np.repeat(image, 3, axis=2)
    return image

def _union_mask(mask_3d, H, W):
    """[H,W,N] -> [H,W] boolean union mask"""
    if mask_3d is None or mask_3d.size == 0:
        return np.zeros((H, W), dtype=bool)
    if mask_3d.ndim == 2:
        return mask_3d.astype(bool)
    return (np.sum(mask_3d, axis=-1) > 0)

def save_overlays_and_metrics(model, dataset, out_dir, score_thresh=0.8, limit=None):
    os.makedirs(out_dir, exist_ok=True)

    # separate folders by TP/FP/FN/TN
    for sub in ["TP", "FP", "FN", "TN"]:
        os.makedirs(os.path.join(out_dir, sub), exist_ok=True)

    ids = dataset.image_ids[:limit] if limit else dataset.image_ids

    TP = FP = FN = TN = 0

    for image_id in tqdm(ids):
        image = dataset.load_image(image_id)
        image = _ensure_rgb3(image)
        H, W = image.shape[:2]

        # GET REAL FILENAME
        image_info = dataset.image_info[image_id]
        filename = os.path.splitext(os.path.basename(image_info["path"]))[0]

        gt_mask, _ = dataset.load_mask(image_id)
        gt_any = bool(gt_mask.size > 0 and np.sum(gt_mask) > 0)
        gt_u = _union_mask(gt_mask, H, W)

        r = model.detect([image], verbose=0)[0]

        # filter by threshold
        if r["scores"].size:
            keep = r["scores"] >= score_thresh
            pred_masks = r["masks"][:, :, keep] if np.any(keep) else np.zeros((H, W, 0), dtype=bool)
        else:
            pred_masks = np.zeros((H, W, 0), dtype=bool)

        pred_any = pred_masks.shape[-1] > 0
        pr_u = _union_mask(pred_masks, H, W)

        # confusion
        if gt_any and pred_any:
            TP += 1; status = "TP"
        elif (not gt_any) and pred_any:
            FP += 1; status = "FP"
        elif gt_any and (not pred_any):
            FN += 1; status = "FN"
        else:
            TN += 1; status = "TN"

        # save overlay image
        fig = plt.figure(figsize=(12, 4))

        ax1 = fig.add_subplot(1, 3, 1)
        ax1.imshow(image); ax1.set_title("Image"); ax1.axis("off")

        ax2 = fig.add_subplot(1, 3, 2)
        ax2.imshow(image)
        ax2.imshow(np.ma.masked_where(~gt_u, gt_u), alpha=0.5)
        ax2.set_title("GT"); ax2.axis("off")

        ax3 = fig.add_subplot(1, 3, 3)
        ax3.imshow(image)
        ax3.set_title(f"Pred ({status})") 
        ax3.axis("off")

        rois = r["rois"]
        masks = r["masks"]
        class_ids = r["class_ids"]
        scores = r["scores"]    

        for i in range(rois.shape[0]):
            if scores[i] < score_thresh:
                continue

            y1, x1, y2, x2 = rois[i]

            # mask overlay
            mask = masks[:, :, i]
            ax3.imshow(np.ma.masked_where(~mask, mask), alpha=0.4)

            # label text (like visualize.display_instances)
            label = dataset.class_names[class_ids[i]]
            score = scores[i]

            ax3.text(
                x1, y1,
                f"{label} {score:.2f}",
                color="white",
                fontsize=8,
                bbox=dict(facecolor="red", alpha=0.5, pad=2)
            )

        plt.tight_layout()
        out_path = os.path.join(out_dir, status, f"{filename}.png")
        
        plt.savefig(out_path, dpi=120)
        plt.close(fig)

    # metrics
    total = TP + FP + FN + TN
    accuracy  = (TP + TN) / total if total else 0.0
    precision = TP / (TP + FP) if (TP + FP) else 0.0
    recall    = TP / (TP + FN) if (TP + FN) else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    print("\nConfusion (image-level presence/absence):")
    print(f"TP={TP}  FP={FP}  FN={FN}  TN={TN}  (Total={total})")

    print("\nMetrics:")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1       : {f1:.4f}")
    print(f"Saved overlays to: {out_dir}")

    return {"TP": TP, "FP": FP, "FN": FN, "TN": TN,
            "accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

In [15]:
out_dir=r"C:\newtest_overlays"

In [16]:
metrics_val = save_overlays_and_metrics(
    model=model_infer,
    dataset=dataset_val,
    out_dir=os.path.join(out_dir, "30_trpre_post1_val"),
    score_thresh=0.8
)

  0%|          | 0/400 [00:00<?, ?it/s]c:\Users\iaadeniran\AppData\Local\ESRI\conda\envs\maskrcnn_env\lib\site-packages\keras\src\engine\training_v1.py:2359: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,
100%|██████████| 400/400 [04:41<00:00,  1.42it/s]


Confusion (image-level presence/absence):
TP=320  FP=0  FN=0  TN=80  (Total=400)

Metrics:
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1       : 1.0000
Saved overlays to: C:\newtest_overlays\30_trpre_post1_val


In [19]:
metrics_val = save_overlays_and_metrics(
    model=model_infer,
    dataset=dataset_val,
    out_dir=os.path.join(out_dir, "30pre_post1_val"),
    score_thresh=0.8
)

100%|██████████| 400/400 [04:23<00:00,  1.52it/s]


Confusion (image-level presence/absence):
TP=320  FP=6  FN=0  TN=74  (Total=400)

Metrics:
Accuracy : 0.9850
Precision: 0.9816
Recall   : 1.0000
F1       : 0.9907
Saved overlays to: C:\newtest_overlays\30pre_post1_val


In [14]:
data_dir = r"C:\oil_well_n\Code\Mask-RCNN_TF2.14.0\datasets\SiteBtest\test_tp"
dataset_test = OilWellDataset()
dataset_test.load_oilwells(data_dir, "test")
dataset_test.prepare()

In [15]:
metrics_val = save_overlays_and_metrics(
    model=model_infer,
    dataset=dataset_test,
    out_dir=os.path.join(out_dir, "30_tpnn"),
    score_thresh=0.8
)

  0%|          | 0/5 [00:00<?, ?it/s]c:\Users\iaadeniran\AppData\Local\ESRI\conda\envs\maskrcnn_env\lib\site-packages\keras\src\engine\training_v1.py:2359: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,
100%|██████████| 5/5 [00:03<00:00,  1.30it/s]


Confusion (image-level presence/absence):
TP=5  FP=0  FN=0  TN=0  (Total=5)

Metrics:
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1       : 1.0000
Saved overlays to: C:\newtest_overlays\30_tpnn


In [28]:
dataset_test = OilWellDataset()
dataset_test.load_oilwells(DATASET_DIR, "test")
dataset_test.prepare()

In [29]:
metrics_val = save_overlays_and_metrics(
    model=model_infer,
    dataset=dataset_test,
    out_dir=os.path.join(out_dir, "pre_post1_test_80"),
    score_thresh=0.8
)

100%|██████████| 202/202 [04:29<00:00,  1.33s/it]


Confusion (image-level presence/absence):
TP=154  FP=16  FN=0  TN=32  (Total=202)

Metrics:
Accuracy : 0.9208
Precision: 0.9059
Recall   : 1.0000
F1       : 0.9506
Saved overlays to: C:\scaled_overlays\30ep\pre_post1_test_80


In [30]:
metrics_val = save_overlays_and_metrics(
    model=model_infer,
    dataset=dataset_test,
    out_dir=os.path.join(out_dir, "pre_post1_test_95"),
    score_thresh=0.95
)

100%|██████████| 202/202 [04:39<00:00,  1.38s/it]


Confusion (image-level presence/absence):
TP=154  FP=1  FN=0  TN=47  (Total=202)

Metrics:
Accuracy : 0.9950
Precision: 0.9935
Recall   : 1.0000
F1       : 0.9968
Saved overlays to: C:\scaled_overlays\30ep\pre_post1_test_95


In [31]:
data_dir = r"C:\oil_well_n\Code\Mask-RCNN_TF2.14.0\datasets\SiteA_post"
dataset_test = OilWellDataset()
dataset_test.load_oilwells(data_dir, "test")
dataset_test.prepare()

In [32]:
metrics_val = save_overlays_and_metrics(
    model=model_infer,
    dataset=dataset_test,
    out_dir=os.path.join(out_dir, "post1_test_95"),
    score_thresh=0.95
)

100%|██████████| 101/101 [02:25<00:00,  1.44s/it]


Confusion (image-level presence/absence):
TP=78  FP=1  FN=0  TN=22  (Total=101)

Metrics:
Accuracy : 0.9901
Precision: 0.9873
Recall   : 1.0000
F1       : 0.9936
Saved overlays to: C:\scaled_overlays\30ep\post1_test_95


In [33]:
data_dir = r"C:\oil_well_n\Code\Mask-RCNN_TF2.14.0\datasets\SiteA_pre"
dataset_test = OilWellDataset()
dataset_test.load_oilwells(data_dir, "test")
dataset_test.prepare()

In [34]:
metrics_val = save_overlays_and_metrics(
    model=model_infer,
    dataset=dataset_test,
    out_dir=os.path.join(out_dir, "pre1_test_95"), 
    score_thresh=0.95
)

100%|██████████| 101/101 [02:29<00:00,  1.48s/it]


Confusion (image-level presence/absence):
TP=76  FP=0  FN=0  TN=25  (Total=101)

Metrics:
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1       : 1.0000
Saved overlays to: C:\scaled_overlays\30ep\pre1_test_95


In [35]:
data_dir = r"C:\oil_well_n\Code\Mask-RCNN_TF2.14.0\datasets\siteB_oilwell"
dataset_test = OilWellDataset()
dataset_test.load_oilwells(data_dir, "test")
dataset_test.prepare()

In [36]:
metrics_val = save_overlays_and_metrics(
    model=model_infer,
    dataset=dataset_test,
    out_dir=os.path.join(out_dir, "post2_test_95"), 
    score_thresh=0.95
)

100%|██████████| 101/101 [02:17<00:00,  1.36s/it]


Confusion (image-level presence/absence):
TP=0  FP=0  FN=79  TN=22  (Total=101)

Metrics:
Accuracy : 0.2178
Precision: 0.0000
Recall   : 0.0000
F1       : 0.0000
Saved overlays to: C:\scaled_overlays\30ep\post2_test_95


In [37]:
metrics_val = save_overlays_and_metrics(
    model=model_infer,
    dataset=dataset_test,
    out_dir=os.path.join(out_dir, "post2_test_80"), 
    score_thresh=0.80
)

100%|██████████| 101/101 [02:15<00:00,  1.34s/it]


Confusion (image-level presence/absence):
TP=2  FP=2  FN=77  TN=20  (Total=101)

Metrics:
Accuracy : 0.2178
Precision: 0.5000
Recall   : 0.0253
F1       : 0.0482
Saved overlays to: C:\scaled_overlays\30ep\post2_test_80


In [38]:
metrics_val = save_overlays_and_metrics(
    model=model_infer,
    dataset=dataset_val,
    out_dir=os.path.join(out_dir, "pre_post1_val"),
    score_thresh=0.8
)

  8%|▊         | 32/400 [00:44<08:34,  1.40s/it]


KeyboardInterrupt: 

In [22]:
metrics_val = save_overlays_and_metrics(
    model=model_infer,
    dataset=dataset_val,
    out_dir=os.path.join(out_dir, "ep30_95/pre_post1_val"),
    score_thresh=0.95
)

100%|██████████| 400/400 [14:18<00:00,  2.15s/it]


Confusion (image-level presence/absence):
TP=320  FP=0  FN=0  TN=80  (Total=400)

Metrics:
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1       : 1.0000
Saved overlays to: C:\scaled_overlays\30ep\ep30_95/pre_post1_val


In [23]:
# data_dir = r"C:\oil_well_n\Code\Mask-RCNN_TF2.14.0\datasets\SiteA_pre"
dataset_test = OilWellDataset()
dataset_test.load_oilwells(DATASET_DIR, "test")
dataset_test.prepare()

In [24]:
metrics_val = save_overlays_and_metrics(
    model=model_infer,
    dataset=dataset_val,
    out_dir=os.path.join(out_dir, "95/pre_post1_val"),
    score_thresh=0.95
)

100%|██████████| 400/400 [14:42<00:00,  2.21s/it]


Confusion (image-level presence/absence):
TP=320  FP=0  FN=0  TN=80  (Total=400)

Metrics:
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1       : 1.0000
Saved overlays to: C:\scaled_overlays\30ep\95/pre_post1_val


In [ ]:
metrics_val = save_overlays_and_metrics(
    model=model_infer,
    dataset=dataset_test,
    out_dir=os.path.join(out_dir, "pre_post1_test_95"),
    score_thresh=0.95
)